# Create your own Xarray Backend
In this lesson, we will learn about and how to create your own custom xarray backend

:::{admonition} Learning Goals
- Learn how to create your own custom backend
- Learn how to use a custom imageio backend to open a GIF
- Learn how to extend the imagegio backend to write out a GIF
:::

## Why should you create your own Xarray backend?

- Allows you to use xarray's interface
- Easy integration with dask for lazy loading with Dask
- You don’t need to integrate any code in Xarray.


## Setting up the BackendEntrypoint

To set up our BackendEntrypoint we can implement a subclass of BackendEntrypoint and expose the `open_dataset` method.

For this example we are going to use a simple `imageio` an image reader and write

In [39]:
from xarray.backends import BackendEntrypoint

class MyBackendEntrypoint(BackendEntrypoint):
    def open_dataset(
        self,
        filename_or_obj,
        *,
        drop_variables=None,
    ):

        return my_open_dataset(filename_or_obj, drop_variables=drop_variables)


In [3]:
import xarray
from advanced.backends.imageio_ import ImageIOBackend
import imageio as iio
import xarray as xr
import numpy as np

class SimpleImageWrtiter(ImageIOBackend):
    def open_dataset(
        self,
        filename_or_obj,
        *,
        drop_variables=None,
    ):
        dims = ['time', 'height', 'width', 'color']
        img_arr = iio.read(filename_or_obj)
        var = xr.Variable(dims=dims, data=img_arr)
        #coords = {"x": np.arange(img_arr.size)}
        coords = {"x": np.arange(img_arr.size)}
        #coords = xr.Coordinates.from_xindex(time).assign(color=['red', 'green', 'blue'])
        return xr.Dataset({"foo": var}, coords=coords)

ModuleNotFoundError: No module named 'advanced'

In [41]:
class BinaryBackend(xr.backends.BackendEntrypoint):
    def open_dataset(
        self,
        filename_or_obj,
        *,
        drop_variables=None,
        # backend specific parameter
        dtype=np.int64,
    ):
        with open(filename_or_obj) as f:
            arr = np.fromfile(f, dtype)

        var = xr.Variable(dims=("x"), data=arr)
        coords = {"x": np.arange(arr.size) * 10}
        return xr.Dataset({"foo": var}, coords=coords)

In [ ]:
from xarray.backends import BackendEntrypoint
import imageio as iio
import xarray as xr
import numpy as np

class SimpleImageReader(BackendEntrypoint):
    import imageio.v3 as iio

        with iio.imopen(filename_or_obj, io_mode="r") as f:
            properties = f.properties()
            metadata = f.metadata()

            dims = ["time", "height", "width", "color"]

            background = metadata["background"]
            duration = metadata["duration"]
            loop = metadata["loop"]

            shape = properties.shape
            dtype = properties.dtype

        if isinstance(duration, (int, float)):
            time_values = np.timedelta64(duration, "ms") * np.arange(shape[0])
        else:
            time_values = np.array(duration, dtype="timedelta64[ms]")

        time = xr.indexes.PandasIndex(pd.Index(time_values), dim="time")

        backend_array = ImageIOBackendArray(
            filename_or_obj=filename_or_obj,
            shape=shape,
            dtype=dtype,
            lock=xr.backends.locks.SerializableLock(),
        )
        data = xr.core.indexing.LazilyIndexedArray(backend_array)

        var = xr.Variable(
            dims=dims,
            data=data,
            attrs={"loop": loop},
            encoding={
                "preferred_chunks": dict(zip(dims, (1, *shape[1:]))),
                "fill_value": background,
            },
        )
        coords = xr.Coordinates.from_xindex(time).assign(color=["red", "green", "blue"])

        return xr.Dataset({"data": var}, coords=coords)

In [ ]:
import xarray as xr
xr.open_dataset()

import xarray asr xr

In [40]:
precip_gif_array = iio.imread(
    "img_data/test_precip.gif"
)

precip_gif_array

/var/folders/gm/pz37_1hs3sv4c_wxjcyhhfnw0000gq/T/ipykernel_57860/714010001.py:1: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  precip_gif_array = iio.imread(


array([[[254, 254, 254],
        [254, 254, 254],
        [254, 254, 254],
        ...,
        [254, 254, 254],
        [254, 254, 254],
        [254, 254, 254]],

       [[254, 254, 254],
        [254, 254, 254],
        [254, 254, 254],
        ...,
        [254, 254, 254],
        [254, 254, 254],
        [254, 254, 254]],

       [[254, 254, 254],
        [254, 254, 254],
        [254, 254, 254],
        ...,
        [254, 254, 254],
        [254, 254, 254],
        [254, 254, 254]],

       ...,

       [[254, 254, 254],
        [254, 254, 254],
        [254, 254, 254],
        ...,
        [254, 254, 254],
        [254, 254, 254],
        [254, 254, 254]],

       [[254, 254, 254],
        [254, 254, 254],
        [254, 254, 254],
        ...,
        [254, 254, 254],
        [254, 254, 254],
        [254, 254, 254]],

       [[254, 254, 254],
        [254, 254, 254],
        [254, 254, 254],
        ...,
        [254, 254, 254],
        [254, 254, 254],
        [254, 254, 254]]

In [38]:
xr.open_dataset('img_data/test_precip.gif', engine=SimpleImageReader)

TypeError: Variable.__init__() missing 1 required positional argument: 'dims'